Load Raw Renewal Calls + Billings (need renewal date)


In [104]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df_rc = pd.read_csv('../../../data/interim/renewal_calls_cleaned.csv')

# Load billings to get prospect_renewal_date per co_ref
df_billings = pd.read_csv('../../../data/interim/billings_features.csv')

# Keep only co_ref and prospect_renewal_date from billings
df_renewal_dates = df_billings[['co_ref', 'renewal_month']].copy()

print(f"Renewal calls shape : {df_rc.shape}")
print(f"Unique customers    : {df_rc['co_ref'].nunique()}")

Renewal calls shape : (153269, 28)
Unique customers    : 36303


Parse Dates

In [105]:
df_rc['call_date'] = pd.to_datetime(df_rc['call_date'], errors='coerce')

# renewal_month in billings is the prospect_renewal_date
df_renewal_dates['prospect_renewal_date'] = pd.to_datetime(
    df_renewal_dates['renewal_month'], errors='coerce'
)
df_renewal_dates.drop(columns=['renewal_month'], inplace=True)

Merge Renewal Date onto Calls

In [106]:
# Each call now knows the customer's renewal date
df_rc = df_rc.merge(df_renewal_dates, on='co_ref', how='left')

print(f"Calls with renewal date matched: {df_rc['prospect_renewal_date'].notna().sum()}")
print(f"Calls with no match            : {df_rc['prospect_renewal_date'].isna().sum()}")

Calls with renewal date matched: 393080
Calls with no match            : 1628


Apply 14-Day Temporal Filter (Model 2 window)

In [107]:
WINDOW_DAYS = 14

# Window: call must be within 14 days before the renewal date
window_start = df_rc['prospect_renewal_date'] - pd.Timedelta(days=WINDOW_DAYS)
window_end   = df_rc['prospect_renewal_date']

mask = (
    (df_rc['call_date'] >= window_start) &
    (df_rc['call_date'] <  window_end)
)

df_rc_14 = df_rc[mask].copy()

print(f"Total calls              : {len(df_rc):,}")
print(f"Calls in 14-day window   : {len(df_rc_14):,}")
print(f"Calls outside window     : {len(df_rc) - len(df_rc_14):,}")
print(f"Customers with 14d calls : {df_rc_14['co_ref'].nunique():,}")

Total calls              : 394,708
Calls in 14-day window   : 1,266
Calls outside window     : 393,442
Customers with 14d calls : 968


Encode Call Direction

In [108]:
# Encode direction
df_rc_14['is_inbound'] = (df_rc_14['call_direction'] == 'IN_BOUND').astype(int)

Encode Yes/No/Unknown Columns

In [109]:
# Yes/No columns
yes_no_cols = [
    'serious_complaint', 'other_complaint', 'discussion_on_price_increase',
    'renewal_impact_due_to_price_increase', 'discount_or_waiver_requested',
    'call_reschedule_request', 'agent_flagged_membership_status_alert',
    'agent_renewal_initiation', 'explicit_competitor_mention',
    'explicit_switching_intent', 'mentioned_competitors',
    'price_switching_mentioned', 'percentage_price_increase_mentioned',
    'monetary_price_increase_mentioned', 'customer_asked_for_justification',
    'discount_offered',
]
for col in yes_no_cols:
    df_rc_14[col] = df_rc_14[col].map({'Yes': 1, 'No': 0,
                                        'Not Discussed': 0, 'Unknown': 0})

Encode Remaining Categorical Columns

In [110]:
# Categorical encodings
df_rc_14['membership_renewal_decision'] = df_rc_14['membership_renewal_decision'].map(
    {'Yes': 1, 'No': 0, 'Unknown': 0})

df_rc_14['desire_to_cancel_encoded'] = df_rc_14['desire_to_cancel'].map(
    {'Renewed': 0, 'Not Discussed': 0, 'Unknown': 0, 'Desired to Cancel': 1})

df_rc_14['customer_response_encoded'] = df_rc_14['customer_response'].map(
    {'Positive': 1, 'Neutral': 0, 'Not Discussed': 0, 'Unknown': 0, 'Negative': -1})

df_rc_14['topic_by_customer'] = (df_rc_14['topic_introduced_by'] == 'Customer').astype(int)
df_rc_14['topic_by_agent']    = (df_rc_14['topic_introduced_by'] == 'Agent').astype(int)

df_rc_14['competitor_better_value']  = (df_rc_14['competitor_value_comparison'] == 'Better Value').astype(int)
df_rc_14['competitor_similar_value'] = (df_rc_14['competitor_value_comparison'] == 'Similar Value').astype(int)
df_rc_14['competitor_better_service']= (df_rc_14['competitor_benefits_mentioned'] == 'Better Service').astype(int)
df_rc_14['competitor_cheaper']       = (df_rc_14['competitor_benefits_mentioned'] == 'Discounts').astype(int)

Handle price_range_mentioned

In [111]:
df_rc_14['price_range_discussed_flag'] = pd.to_numeric(df_rc_14['price_range_mentioned'], errors='coerce').notna().astype(int)

Feature Creation

In [112]:
# Engineered features
df_rc_14['rc_high_risk_call'] = (
    (df_rc_14['desire_to_cancel_encoded']    == 1) |
    (df_rc_14['serious_complaint']           == 1) |
    (df_rc_14['explicit_switching_intent']   == 1) |
    (df_rc_14['explicit_competitor_mention'] == 1)
).astype(int)

df_rc_14['competitor_threat_flag'] = (
    (df_rc_14['mentioned_competitors']   == 1) &
    (df_rc_14['competitor_better_value'] == 1)
).astype(int)

df_rc_14['price_pressure_flag'] = (
    (df_rc_14['discussion_on_price_increase']        == 1) &
    (df_rc_14['renewal_impact_due_to_price_increase']== 1)
).astype(int)

df_rc_14['complaint_count']         = df_rc_14['serious_complaint'] + df_rc_14['other_complaint']
df_rc_14['competitor_signal_count'] = (
    df_rc_14['explicit_competitor_mention'] +
    df_rc_14['mentioned_competitors'] +
    df_rc_14['price_switching_mentioned'] +
    df_rc_14['competitor_better_value']
)

df_rc_14['agent_active_retention'] = (
    (df_rc_14['agent_renewal_initiation']              == 1) |
    (df_rc_14['discount_offered']                      == 1) |
    (df_rc_14['agent_flagged_membership_status_alert'] == 1)
).astype(int)

Aggregate per customer

In [113]:
agg_dict = {
    'call_id'  : 'count',
    'is_inbound' : 'sum',
    'call_number' : 'max',
    'membership_renewal_decision' : 'max',
    'desire_to_cancel_encoded' : 'max',
    'customer_response_encoded' : 'mean',
    'serious_complaint' : 'max',
    'other_complaint' : 'max',
    'complaint_count' : 'sum',
    'discussion_on_price_increase': 'max',
    'renewal_impact_due_to_price_increase' : 'max',
    'discount_or_waiver_requested': 'max',
    'discount_offered' : 'max',
    'price_pressure_flag' : 'max',
    'price_range_discussed_flag' : 'max',
    'percentage_price_increase_mentioned'  : 'max',
    'monetary_price_increase_mentioned' : 'max',
    'explicit_competitor_mention' : 'max',
    'explicit_switching_intent' : 'max',
    'mentioned_competitors': 'max',
    'price_switching_mentioned' : 'max',
    'competitor_better_value': 'max',
    'competitor_similar_value' : 'max',
    'competitor_better_service' : 'max',
    'competitor_cheaper' : 'max',
    'competitor_signal_count': 'sum',
    'competitor_threat_flag' : 'max',
    'agent_renewal_initiation' : 'max',
    'agent_flagged_membership_status_alert': 'max',
    'agent_active_retention' : 'max',
    'call_reschedule_request' : 'max',
    'topic_by_customer' : 'max',
    'topic_by_agent' : 'max',
    'customer_asked_for_justification' : 'max',
    'rc_high_risk_call' : 'max',
}

df_rc_agg = df_rc_14.groupby('co_ref').agg(agg_dict).reset_index()

df_rc_agg.rename(columns={
    'call_id'    : 'rc_14d_total_calls',
    'call_number': 'rc_14d_max_call_number',
    'is_inbound' : 'rc_14d_inbound_calls',
}, inplace=True)

df_rc_agg['rc_14d_outbound_calls'] = (
    df_rc_agg['rc_14d_total_calls'] - df_rc_agg['rc_14d_inbound_calls']
)

Drop Columns No Longer Needed

In [114]:
drop_cols = [
    'call_direction', # replaced by is_inbound
    'desire_to_cancel', # replaced by desire_to_cancel_encoded
    'customer_response', # replaced by customer_response_encoded
    'topic_introduced_by', # replaced by topic_by_customer / topic_by_agent
    'competitor_value_comparison',  # replaced by competitor_better_value / similar_value
    'competitor_benefits_mentioned',# replaced by competitor_better_service / cheaper
    'price_range_mentioned',     # replaced by price_range_value + flag
]

df_rc_agg.drop(columns=[c for c in drop_cols if c in df_rc_agg.columns], inplace=True)

In [115]:

print(f"Shape: {df_rc_agg.shape}")
print(f"Unique customers: {df_rc_agg['co_ref'].nunique()}")

df_rc_agg.to_csv('../../../data/interim/renewal_calls_features.csv', index=False)

Shape: (968, 37)
Unique customers: 968
